In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import seaborn as sns
import os
import matplotlib.dates as mdates
import cv2

In [2]:
bin_specs = { 
    "small":
            {
                "sizerange": (0.125,4), #cm^2
                "logrange": (-3.0,2.0), #log(2) units
                "count_col": "count_small",
                "bm_col": "bm_small", 
                "color": "blue",
                "avg_mass": 2.75,
                "class": "small"
            },
    
    "medium-small":
            {
                "sizerange": (4,12), #cm^2
                "logrange": (2.0,3.58),
                "count_col": "count_medium-small",
                "bm_col": "bm_medium-small",
                "color": "orange",
                "avg_mass": 54,
                "class": "medium-small"
            },
    "medium-large":
            {
                "sizerange" : (12,55), #cm^2
                "logrange" : (3.58, 5.78),
                "count_col": "count_medium-large",
                "bm_col": "bm_medium-large",
                "color": "purple",
                "avg_mass": 361,
                "class" : "medium-large"
            },
    "large":
            {
                "sizerange": (55,256), #cm^2
                "logrange": (5.78,8.0),
                "count_col": "count_large",
                "bm_col": "bm_large",
                "color": "skyblue",
                "avg_mass": 418,
                "class": "large"
            }
}

In [5]:
predictions23_ws1 = pd.read_csv("/Users/suhavi./Projects/moths/The Project/Results/InferencePipeline/WS1_2023_sizeclass_predictions.csv")

In [6]:
predictions23_ws1

,Unnamed: 0,filePath_pred,fileName,year_pred,site_pred,X_Min,Y_Min,X_Max,Y_Max,d_X,...,megapixels,filePath.orig,fileName.orig,avg_red,avg_green,avg_blue,fileName_old,size_log2,timebin,size_bin
0,10568,WS1/2023_WS1_0611_0040_WSCT3029.JPG,2023_WS1_0611_0040_WSCT3029.JPG,2023,WS1,2671.53780,2326.86740,2762.50980,2397.03660,90.971924,...,20.79,./2023/WS1/101_WSCT/WSCT3029.JPG,WSCT3029.JPG,0.573,0.715,0.852,2023_WS1_0611_40_WSCT3029.JPG,0.174473,0,small
1,11846,WS1/2023_WS1_0611_0125_WSCT3038.JPG,2023_WS1_0611_0125_WSCT3038.JPG,2023,WS1,1162.74880,716.36970,1415.21730,975.74884,252.468500,...,20.79,./2023/WS1/101_WSCT/WSCT3038.JPG,WSCT3038.JPG,0.569,0.716,0.847,2023_WS1_0611_125_WSCT3038.JPG,3.533236,1,medium-small
2,11847,WS1/2023_WS1_0611_0125_WSCT3038.JPG,2023_WS1_0611_0125_WSCT3038.JPG,2023,WS1,2787.02120,778.09200,3014.83940,916.30350,227.818120,...,20.79,./2023/WS1/101_WSCT/WSCT3038.JPG,WSCT3038.JPG,0.569,0.716,0.847,2023_WS1_0611_125_WSCT3038.JPG,2.476830,1,medium-small
3,11848,WS1/2023_WS1_0611_0125_WSCT3038.JPG,2023_WS1_0611_0125_WSCT3038.JPG,2023,WS1,5265.16060,722.60490,5500.14400,878.58870,234.983400,...,20.79,./2023/WS1/101_WSCT/WSCT3038.JPG,WSCT3038.JPG,0.569,0.716,0.847,2023_WS1_0611_125_WSCT3038.JPG,2.696025,1,medium-small
4,11849,WS1/2023_WS1_0611_0125_WSCT3038.JPG,2023_WS1_0611_0125_WSCT3038.JPG,2023,WS1,2589.25540,805.57874,2756.05790,939.00433,166.802490,...,20.79,./2023/WS1/101_WSCT/WSCT3038.JPG,WSCT3038.JPG,0.569,0.716,0.847,2023_WS1_0611_125_WSCT3038.JPG,1.976246,1,small
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30324,13093,WS1/2023_WS1_0930_0030_WSCT4263.JPG,2023_WS1_0930_0030_WSCT4263.JPG,2023,WS1,5388.73240,2809.32230,5593.84380,2985.73100,205.111330,...,20.79,./2023/WS1/101_WSCT/WSCT4263.JPG,WSCT4263.JPG,0.404,0.573,0.783,2023_WS1_0930_30_WSCT4263.JPG,2.677399,0,medium-small
30325,13332,WS1/2023_WS1_0930_0145_WSCT4278.JPG,2023_WS1_0930_0145_WSCT4278.JPG,2023,WS1,424.88058,464.43512,565.08734,643.50757,140.206760,...,20.79,./2023/WS1/101_WSCT/WSCT4278.JPG,WSCT4278.JPG,0.408,0.576,0.788,2023_WS1_0930_145_WSCT4278.JPG,2.150169,1,medium-small
30326,5706,WS1/2023_WS1_0930_0240_WSCT4289.JPG,2023_WS1_0930_0240_WSCT4289.JPG,2023,WS1,5364.34900,2817.33600,5609.59000,3002.08150,245.240720,...,20.79,./2023/WS1/101_WSCT/WSCT4289.JPG,WSCT4289.JPG,0.434,0.595,0.801,2023_WS1_0930_240_WSCT4289.JPG,3.001809,2,medium-small
30327,24134,WS1/2023_WS1_0930_2245_WSCT4242.JPG,2023_WS1_0930_2245_WSCT4242.JPG,2023,WS1,4135.61870,2268.21040,4345.21900,2449.50830,209.600590,...,20.79,./2023/WS1/101_WSCT/WSCT4242.JPG,WSCT4242.JPG,0.404,0.574,0.787,2023_WS1_0930_2245_WSCT4242.JPG,2.748075,22,medium-small


In [7]:
from matplotlib import colors
import matplotlib.pyplot as plt
from pathlib import Path

IMGPATH = "/Users/suhavi./Projects/moths/The Project/EDI/2023"
OUTDIR = Path("/Users/suhavi./Projects/moths/The Project/Results/InferencePipeline/predictions_WS1_23")
OUTDIR.mkdir(parents=True, exist_ok=True)

groups = predictions23_ws1.groupby("filePath_pred")

for name, grdf in groups:
    img = cv2.imread(os.path.join(IMGPATH,name))

    if img is None:
        print(f"Could not read {img}")
        continue

    for i, row in grdf.iterrows():
        rgb = colors.to_rgb(bin_specs[row["size_bin"]]["color"])
        bgr = (int(rgb[2]*255), int(rgb[1]*255), int(rgb[0]*255))
        # print(colour)
        cv2.rectangle(
            img, 
            (int(row["X_Min"]),int(row["Y_Min"])), 
            (int(row["X_Max"]), int(row["Y_Max"])),
            bgr,
            3
        )
    outname = Path(name).name + ".png"
    cv2.imwrite(str(OUTDIR / outname), img)
    # plt.savefig("/Users/suhavi./Projects/moths/The Project/Results/InferencePipeline/predictions/"+name+".png", dpi=300, bbox_inches='tight')
    # break